<a href="https://colab.research.google.com/github/normanli33/531w20-Time-Series-Analysis/blob/master/notebook_Titanic_Forest_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## The competition is simple: use machine learning to create a model that predicts which passengers survived the Titanic shipwreck.


In [ ]:
# pd.options.display.max_columns = 25

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
d_train = pd.read_csv("/kaggle/input/titanic/train.csv")

In [ ]:
d_train.head(5)

In [ ]:
d_train.hist(figsize=(12, 8))
plt.tight_layout()

In [ ]:
# This sums up all the 'True' values (True = 1, False = 0)
missing_count = d_train.isna().sum()

print("Missing count =", missing_count)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Creates a heatmap where yellow/bright lines represent missing values
sns.heatmap(d_train.isna(), yticklabels=False, cbar=False, cmap='viridis')
plt.show()

In [ ]:
# class Selector(BaseEstimator, TransformerMixin):  # Added 'class' keyword
#     def __init__(self, columns):
#         self.columns = columns

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         return X[self.columns]  # Just select desired columns directly



In [ ]:
d_train.info()

In [ ]:
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked"]

In [ ]:
num_pipe = Pipeline( steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline( steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer(
    transformers = [
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

In [ ]:
preprocess

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=16,
    n_jobs=-1
)

knn_clf = KNeighborsClassifier(n_neighbors=4)

lr_clf = LogisticRegression(max_iter=1000)

svm_clf= SVC()

voting_clf = VotingClassifier(
    estimators = [('lr', lr_clf), ('svc', svm_clf),('knn', knn_clf)],
    voting = 'hard'
)

pipe = Pipeline( steps=[
    ("preprocess", preprocess,),
    ("model", svm_clf)
])


In [ ]:
pipe

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=16,
    n_jobs=-1
)

knn_clf = KNeighborsClassifier(n_neighbors=4)

lr_clf = LogisticRegression(max_iter=1000)

svm_clf= SVC()

voting_clf = VotingClassifier(
    estimators = [
        ('lr', lr_clf),
        ('svc', svm_clf),
        ('knn', knn_clf),
        ('rf', rf_clf)],
    voting = 'hard'
)

pipe = Pipeline( steps=[
    ("preprocess", preprocess,),
    ("model", voting_clf)
])


### Data source

In [ ]:
# X & y
X_train = d_train.drop("Survived", axis=1)
y_train = d_train["Survived"]

# split data set
X_train, X_test, y_train,y_test= train_test_split(
    X_train, y_train,test_size=0.2,random_state=16,stratify=y_train
)

In [ ]:
# Training
pipe.fit(X_train, y_train)

In [ ]:
#3. Test set predict

y_pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
#4. Evaluate
print("Train accuracy:", pipe.score(X_train, y_train))
print("Test accuracy:", pipe.score(X_test, y_test))

print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

In [ ]:
models = (lr_clf, knn_clf, svm_clf, rf_clf)

for clf in models:
    pipe_clf = Pipeline (steps = [
        ("preprocess", preprocess),
        ("model", clf)
    ])
    pipe_clf.fit(X_train, y_train)
    y_pred = pipe_clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

### the best model is Random Forest Tree
LogisticRegression 0.8100558659217877  
KNeighborsClassifier 0.8212290502793296  
SVC 0.8268156424581006  
RandomForestClassifier 0.8547486033519553  

In [ ]:
feature_names = pipe.named_steps["preprocess"].get_feature_names_out()

print("features: ", feature_names)

In [ ]:
d_tst = pd.read_csv("/kaggle/input/titanic/test.csv")
d_gender = pd.read_csv("/kaggle/input/titanic/gender_submission.csv")

In [ ]:
d_gender.sample(5)

In [ ]:
d_tst.sample(5)

In [ ]:
tst_predict = pipe.predict(d_tst)

print(tst_predict)

In [ ]:
d_tst["Survived"] = tst_predict
submission = d_tst[["PassengerId", "Survived"]]

In [ ]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

import os
assert "submission.csv" in os.listdir("/kaggle/working")
print("OK - submission.csv created in /kaggle/working/")

In [ ]:
os.listdir("/kaggle/working")